# PDHG Tail-Length and \(\tau_0\) Ablations From an Exact Reference Run (Colab)

This notebook runs a sequential, strictly one-factor-at-a-time ablation around the supplied phase-retrieval configuration. It studies only:

1. Tail length \(K_{\rm tail}\).
2. The primal-step scale \(\tau_0\).

The reference candidate is reproduced exactly:

\[
K_{\rm effective}=472,\quad s=419,\quad K_{\rm tail}=53,\quad
c=33.8025084713,\quad
\lambda=2.0103451448,\quad
p_\rho=1.1,\quad
\tau_0=0.01.
\]

The tail-shape parameters \(c\) and \(p_\rho\) are fixed to their reference values in every candidate. They are recorded for auditing but are never ablated or combined with another intervention.

Every run uses the same 100 FFHQ images, batch size 100, seed 99, model, operator, measurements, and evaluation metrics.

## Controlled-ablation design

Tail-length variants keep the switch fixed at \(s=419\). Their effective run length is

\[
K=s+K_{\rm tail}.
\]

Consequently, \(K_{\rm tail}=81\) extends the reference from \(k=471\) through \(k=499\), while reproducing every shared \(\sigma,\tau,\rho,\gamma\) schedule value. The phase-\(\gamma\) anchor remains fixed at \(k=372\).

The supplied run uses an explicit \(\lambda\). Therefore changing only the configured prefix \(\tau_0\) would leave tail \(\tau_k=\sigma_k^2/\lambda\) unchanged. The default policy, scale_full_tau_schedule, uses

\[
\lambda_{\rm variant}
=\lambda_{\rm ref}\frac{\tau_{0,\rm ref}}{\tau_{0,\rm variant}},
\]

so the complete \(\tau_k\) schedule scales with \(\tau_0\), while the reference remains exact. Select prefix_only_fixed_lambda only if you deliberately want to change the prefix field and leave the tail tau unchanged.

No Cartesian mode is provided: every non-reference candidate changes exactly one of the two requested conceptual quantities.

In [ ]:
#@title Project and ablation settings

SETUP_MODE = "git"  #@param ["git", "drive_zip"]
REPO_URL = "https://github.com/Seif-Hussein/dyscode.git"  #@param {type:"string"}
REPO_BRANCH = "codex-pdhg-colab-light-100"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/mycode2.zip"  #@param {type:"string"}

REPO_DIR = "/content/mycode2"  #@param {type:"string"}
PYTHON_BIN = "/usr/bin/python3"  #@param {type:"string"}
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/pdhg_tail_tau0_ablation_exports"  #@param {type:"string"}
DRIVE_FFHQ_DATA_DIR = "/content/drive/MyDrive/mycode/test-ffhq"  #@param {type:"string"}
SESSION_TAG = ""  #@param {type:"string"}
STUDY_NAME = "Inverse_PR_TailLength_Tau0_Ablation"  #@param {type:"string"}
CONFIG_NAME = "default_ffhq.yaml"  #@param {type:"string"}

SEED = 99  #@param {type:"integer"}
TOTAL_IMAGES = 100  #@param {type:"integer"}
BATCH_SIZE = 100  #@param {type:"integer"}
DATA_START_IDX = 0  #@param {type:"integer"}
MEASUREMENT_SIGMA = 0.05  #@param {type:"number"}

# Exact reference schedule. Tail c and rho power remain fixed in all candidates.
REFERENCE_MAX_ITER = 500
REFERENCE_EFFECTIVE_K = 472
REFERENCE_TAIL_STEPS = 53
REFERENCE_TAIL_C = 33.8025084713
REFERENCE_TAIL_LAMBDA = 2.0103451448
REFERENCE_RHO_POWER = 1.1
REFERENCE_TAU0 = 0.01
REFERENCE_SIGMA_MAX = 10.0
REFERENCE_SIGMA_MIN = 0.075
REFERENCE_PREFIX_NUM_STEPS = 500
REFERENCE_PREFIX_TIMESTEP = "poly-7"
REFERENCE_GAMMA0 = 1100.0
REFERENCE_GAMMA_ANCHOR = 372

# Only these two quantities are ablated. Reference values are deduplicated automatically.
TAIL_STEPS_VALUES = "10;30;53;81"  #@param {type:"string"}
TAU0_VALUES = "0.005;0.01;0.02"  #@param {type:"string"}

TAU_ABLATION_POLICY = "scale_full_tau_schedule"  #@param ["scale_full_tau_schedule", "prefix_only_fixed_lambda"]
PRIMARY_METRIC = "psnr"  #@param ["psnr", "ssim", "lpips"]
MAX_CANDIDATES = 0  #@param {type:"integer"}
LOG_TAIL_LINES = 160  #@param {type:"integer"}

SAVE_SAMPLES = False
SAVE_TRAJ = False
SAVE_TRAJ_RAW_DATA = False
EXTRA_BASE_OVERRIDES = ""  #@param {type:"string"}

In [ ]:
#@title Build and preview the strictly one-factor-at-a-time candidates
import json
import math

def parse_number_list(text, cast=float):
    values = []
    for token in str(text).replace(",", ";").split(";"):
        token = token.strip()
        if token:
            values.append(cast(float(token)) if cast is int else cast(token))
    if not values:
        raise ValueError("Every ablation value list must contain at least one number.")
    return values

def close(a, b):
    return math.isclose(float(a), float(b), rel_tol=1e-10, abs_tol=1e-12)

tail_steps_values = parse_number_list(TAIL_STEPS_VALUES, int)
tau0_values = parse_number_list(TAU0_VALUES, float)

reference_switch = int(REFERENCE_EFFECTIVE_K) - int(REFERENCE_TAIL_STEPS)
if reference_switch != 419:
    raise ValueError(f"Expected the supplied reference switch to be 419, got {reference_switch}.")

def sigma_prefix_at(k):
    p = int(str(REFERENCE_PREFIX_TIMESTEP).split("-", 1)[1])
    r = min(k, int(REFERENCE_PREFIX_NUM_STEPS) - 1) / float(int(REFERENCE_PREFIX_NUM_STEPS) - 1)
    hi = float(REFERENCE_SIGMA_MAX) ** (1.0 / p)
    lo = float(REFERENCE_SIGMA_MIN) ** (1.0 / p)
    return ((1.0 - r) * hi + r * lo) ** p

reference_sigma_s = sigma_prefix_at(reference_switch)
reference_implicit_lambda = reference_sigma_s ** 2 / float(REFERENCE_TAU0)

def lambda_for_tau(tau0):
    tau0 = float(tau0)
    if TAU_ABLATION_POLICY == "scale_full_tau_schedule":
        return float(REFERENCE_TAIL_LAMBDA) * float(REFERENCE_TAU0) / tau0
    if TAU_ABLATION_POLICY == "prefix_only_fixed_lambda":
        return float(REFERENCE_TAIL_LAMBDA)
    raise ValueError(f"Unsupported TAU_ABLATION_POLICY: {TAU_ABLATION_POLICY}")

def build_variant(name, tail_steps, tau0, ablated_quantity):
    tail_steps = int(tail_steps)
    tau0 = float(tau0)
    tail_c = float(REFERENCE_TAIL_C)
    rho_power = float(REFERENCE_RHO_POWER)
    effective_k = reference_switch + tail_steps
    if tail_steps < 2:
        raise ValueError("Tail lengths must be at least 2.")
    if effective_k > int(REFERENCE_MAX_ITER):
        raise ValueError(
            f"TAIL_STEPS={tail_steps} requires K={effective_k}, exceeding "
            f"REFERENCE_MAX_ITER={REFERENCE_MAX_ITER}."
        )
    if tau0 <= 0:
        raise ValueError("TAU0 values must be positive.")
    if ablated_quantity not in {"reference", "tail_steps", "tau0"}:
        raise ValueError(f"Unexpected ablated quantity: {ablated_quantity}")

    tail_lambda = lambda_for_tau(tau0)
    gamma_tail_steps = effective_k - int(REFERENCE_GAMMA_ANCHOR)
    if gamma_tail_steps <= 0:
        raise ValueError("A candidate ends before the fixed phase-gamma anchor.")

    sigma = [sigma_prefix_at(k) for k in range(effective_k)]
    tau = [tau0] * effective_k
    rho = list(sigma)
    gamma = [float(REFERENCE_GAMMA0)] * effective_k
    sigma_switch = sigma[reference_switch]

    for k in range(reference_switch, effective_k):
        t = k - reference_switch
        ratio = math.sqrt(tail_c / (tail_c + t))
        sigma[k] = sigma_switch * ratio
        tau[k] = sigma[k] ** 2 / tail_lambda
        rho[k] = sigma_switch * ratio ** rho_power

    gamma_anchor_sigma = sigma[int(REFERENCE_GAMMA_ANCHOR)]
    for k in range(int(REFERENCE_GAMMA_ANCHOR), effective_k):
        gamma[k] = float(REFERENCE_GAMMA0) * (gamma_anchor_sigma / sigma[k]) ** 2

    params = {
        "++inverse_task.admm_config.early_stop": effective_k,
        "sampler.annealing_scheduler_config.theorem1_tail_steps": tail_steps,
        "sampler.annealing_scheduler_config.theorem1_tail_c": tail_c,
        "sampler.annealing_scheduler_config.theorem1_tail_lambda": tail_lambda,
        "sampler.annealing_scheduler_config.theorem1_tail_rho_power": rho_power,
        "inverse_task.admm_config.pdhg.tau": tau0,
        "++inverse_task.admm_config.pdhg.sigma_dual_schedule_tail_steps": gamma_tail_steps,
    }
    summary = {
        "name": name,
        "ablated_quantity": ablated_quantity,
        "effective_k": effective_k,
        "switch": reference_switch,
        "tail_steps": tail_steps,
        "fixed_tail_c": tail_c,
        "fixed_rho_power": rho_power,
        "tau0": tau0,
        "tail_lambda": tail_lambda,
        "gamma_anchor": effective_k - gamma_tail_steps,
        "sigma_switch": sigma_switch,
        "tau_switch": tau[reference_switch],
        "sigma_final": sigma[-1],
        "tau_final": tau[-1],
        "rho_final": rho[-1],
        "gamma_final": gamma[-1],
    }
    return {
        "name": name,
        "params": params,
        "summary": summary,
        "sigma": sigma,
        "tau": tau,
        "rho": rho,
        "gamma": gamma,
    }

variant_specs = [(
    "reference",
    int(REFERENCE_TAIL_STEPS),
    float(REFERENCE_TAU0),
    "reference",
)]
for value in tail_steps_values:
    if int(value) != int(REFERENCE_TAIL_STEPS):
        variant_specs.append((
            f"tail_steps_{int(value)}",
            int(value),
            float(REFERENCE_TAU0),
            "tail_steps",
        ))
for value in tau0_values:
    if not close(value, REFERENCE_TAU0):
        variant_specs.append((
            f"tau0_{float(value):.6g}",
            int(REFERENCE_TAIL_STEPS),
            float(value),
            "tau0",
        ))

variant_plans = [
    build_variant(name, tail_steps, tau0, ablated_quantity)
    for name, tail_steps, tau0, ablated_quantity in variant_specs
]
if int(MAX_CANDIDATES) > 0:
    variant_plans = variant_plans[:int(MAX_CANDIDATES)]
if not variant_plans:
    raise ValueError("No candidates remain after applying MAX_CANDIDATES.")

reference_plan = variant_plans[0]
reference_summary = reference_plan["summary"]
reference_checks = {
    "effective_k": (reference_summary["effective_k"], 472),
    "switch": (reference_summary["switch"], 419),
    "sigma_switch": (reference_summary["sigma_switch"], 0.2147913946345734),
    "tail_lambda": (reference_summary["tail_lambda"], 2.0103451448),
    "fixed_rho_power": (reference_summary["fixed_rho_power"], 1.1),
    "tau0": (reference_summary["tau0"], 0.01),
}
for key, (actual, expected) in reference_checks.items():
    if not close(actual, expected):
        raise RuntimeError(f"Reference mismatch for {key}: {actual} != {expected}")

for item in variant_plans[1:]:
    summary = item["summary"]
    if not close(summary["fixed_tail_c"], REFERENCE_TAIL_C):
        raise RuntimeError("A candidate changed the fixed tail c.")
    if not close(summary["fixed_rho_power"], REFERENCE_RHO_POWER):
        raise RuntimeError("A candidate changed the fixed rho power.")
    if summary["ablated_quantity"] == "tail_steps" and not close(summary["tau0"], REFERENCE_TAU0):
        raise RuntimeError("A tail-length candidate also changed tau0.")
    if summary["ablated_quantity"] == "tau0" and summary["tail_steps"] != REFERENCE_TAIL_STEPS:
        raise RuntimeError("A tau0 candidate also changed tail length.")

print(json.dumps({
    "reference_switch": reference_switch,
    "reference_sigma_s": reference_sigma_s,
    "reference_implicit_lambda": reference_implicit_lambda,
    "explicit_reference_lambda": REFERENCE_TAIL_LAMBDA,
    "fixed_tail_c": REFERENCE_TAIL_C,
    "fixed_rho_power": REFERENCE_RHO_POWER,
    "tau_ablation_policy": TAU_ABLATION_POLICY,
    "candidate_count": len(variant_plans),
}, indent=2))

try:
    import pandas as pd
    display(pd.DataFrame([item["summary"] for item in variant_plans]))
except ImportError:
    for item in variant_plans:
        print(item["summary"])

try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
    for item in variant_plans:
        k_values = list(range(len(item["sigma"])))
        for ax, key in zip(axes, ["sigma", "tau", "rho"]):
            ax.plot(k_values, item[key], label=item["name"], alpha=0.85)
    for ax, title in zip(axes, ["sigma_k", "tau_k", "rho_k"]):
        ax.axvline(reference_switch, color="black", linestyle="--", alpha=0.5)
        ax.set_title(title)
        ax.set_xlabel("iteration k")
        ax.grid(alpha=0.2)
    axes[-1].legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.show()
except ImportError:
    print("matplotlib is unavailable; skipping plots.")

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#@title Fetch the repository
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
repo_dir.parent.mkdir(parents=True, exist_ok=True)
os.chdir(repo_dir.parent)

if repo_dir.exists():
    shutil.rmtree(repo_dir)

if SETUP_MODE == "git":
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, repo_dir.as_posix()],
        check=True,
    )
elif SETUP_MODE == "drive_zip":
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(repo_dir.parent)
    extracted_root = repo_dir.parent / zip_path.stem
    if extracted_root.exists() and extracted_root != repo_dir:
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
        extracted_root.rename(repo_dir)
else:
    raise ValueError(f"Unsupported SETUP_MODE: {SETUP_MODE}")

os.chdir(repo_dir)
print(f"Repository ready: {repo_dir}")

In [ ]:
#@title Install Colab-safe dependencies
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run([PYTHON_BIN, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("Installed requirements-colab.txt")

In [ ]:
#@title Download the FFHQ checkpoint if needed
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
checkpoint_path = Path("pretrained-models/ffhq_10m.pt")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

if checkpoint_path.exists():
    print(f"Checkpoint already present: {checkpoint_path}")
else:
    subprocess.run(
        ["gdown", "--id", "1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh", "-O", checkpoint_path.as_posix()],
        check=True,
    )
    print(f"Downloaded checkpoint to: {checkpoint_path}")

In [ ]:
#@title Write the exact-list ablation configuration
import json
import os
import time
from pathlib import Path

import yaml

os.chdir(REPO_DIR)
repo_dir = Path(REPO_DIR)
drive_data_dir = Path(DRIVE_FFHQ_DATA_DIR)
if not drive_data_dir.exists():
    raise FileNotFoundError(f"FFHQ dataset path not found: {drive_data_dir}")

def parse_override_list(text):
    return [
        item.strip()
        for item in str(text).replace("\n", ";").split(";")
        if item.strip()
    ]

study_tag = SESSION_TAG.strip() or time.strftime("%Y%m%d-%H%M%S")
study_slug = f"{STUDY_NAME}_{study_tag}"
generated_config_path = repo_dir / "tuning" / f"generated_tail_tau0_{study_tag}.yaml"
study_output_root = repo_dir / "tuning_runs" / "colab_tail_tau0"
launcher_log_path = repo_dir / "tuning_runs" / f"{study_slug}.launcher.log"
launcher_pid_path = repo_dir / "tuning_runs" / f"{study_slug}.launcher.pid"
data_end_idx = int(DATA_START_IDX) + int(TOTAL_IMAGES)

base_overrides = [
    "sampler=edm_pdhg",
    "inverse_task=phase_retrieval",
    f"name={study_slug}",
    f"seed={int(SEED)}",
    "gpu=0",
    "wandb=false",
    "show_config=false",
    "show_eval=true",
    f"save_samples={'true' if SAVE_SAMPLES else 'false'}",
    f"save_traj={'true' if SAVE_TRAJ else 'false'}",
    f"save_traj_raw_data={'true' if SAVE_TRAJ_RAW_DATA else 'false'}",
    f"total_images={int(TOTAL_IMAGES)}",
    f"batch_size={int(BATCH_SIZE)}",
    "num_runs=1",
    f"eval_fn_list=[psnr,ssim,lpips]",
    f"data.image_root_path={drive_data_dir.as_posix()}",
    f"data.start_idx={int(DATA_START_IDX)}",
    f"data.end_idx={data_end_idx}",
    f"inverse_task.operator.sigma={float(MEASUREMENT_SIGMA)}",
    "inverse_task.operator.oversample=2.0",
    f"inverse_task.admm_config.max_iter={int(REFERENCE_MAX_ITER)}",
    "++inverse_task.admm_config.denoise.ac_noise=true",
    "inverse_task.admm_config.denoise.final_step=tweedie",
    "inverse_task.admm_config.denoise.lgvd.num_steps=0",
    "inverse_task.admm_config.pdhg.sigma_dual=1100",
    "++inverse_task.admm_config.pdhg.sigma_dual_schedule_mode=to_infinity",
    "++inverse_task.admm_config.pdhg.sigma_dual_schedule_scope=tail",
    "++inverse_task.admm_config.pdhg.sigma_dual_schedule_power=2",
    "++inverse_task.admm_config.pdhg.sigma_dual_schedule_min=1e-8",
    "++inverse_task.admm_config.pdhg.phase_dual_active_y_eps=0.001",
    "++inverse_task.admm_config.pdhg.phase_pr_alpha=0.25",
    "inverse_task.admm_config.pdhg.use_box_proj=false",
    "inverse_task.admm_config.pdhg.use_operator_complex=true",
    "inverse_task.admm_config.pdhg.fft_norm=ortho",
    "inverse_task.admm_config.pdhg.init_dual=noise",
    "inverse_task.admm_config.pdhg.theta_schedule.activate=true",
    "inverse_task.admm_config.pdhg.theta_schedule.start=0",
    "inverse_task.admm_config.pdhg.theta_schedule.end=0",
    "inverse_task.admm_config.pdhg.theta_schedule.warmup=50",
    f"sampler.annealing_scheduler_config.num_steps={int(REFERENCE_PREFIX_NUM_STEPS)}",
    f"sampler.annealing_scheduler_config.sigma_max={float(REFERENCE_SIGMA_MAX)}",
    f"sampler.annealing_scheduler_config.sigma_min={float(REFERENCE_SIGMA_MIN)}",
    "sampler.annealing_scheduler_config.sigma_final=0",
    "sampler.annealing_scheduler_config.schedule=linear",
    f"sampler.annealing_scheduler_config.timestep={REFERENCE_PREFIX_TIMESTEP}",
    "sampler.annealing_scheduler_config.theorem1_tail_sigma_mode=theorem1",
    "sampler.annealing_scheduler_config.theorem1_tail_rho_scale=1.0",
    "sampler.diffusion_scheduler_config.num_steps=10",
    "sampler.diffusion_scheduler_config.sigma_min=0.01",
    "sampler.diffusion_scheduler_config.sigma_final=0",
    "sampler.diffusion_scheduler_config.schedule=linear",
    "sampler.diffusion_scheduler_config.timestep=poly-7",
]
base_overrides.extend(parse_override_list(EXTRA_BASE_OVERRIDES))

tuning_config = {
    "study": {
        "name": study_slug,
        "output_root": study_output_root.as_posix(),
        "result_root": (repo_dir / "results" / "tuning").as_posix(),
        "hydra_root": (repo_dir / "outputs" / "tuning").as_posix(),
    },
    "runner": {
        "python_executable": PYTHON_BIN,
        "entrypoint": "recover_inverse2.py",
        "config_name": CONFIG_NAME,
        "workdir": repo_dir.as_posix(),
    },
    "scoring": {
        "primary_metric": PRIMARY_METRIC,
        "comparator": "auto",
        "aggregate": "mean",
    },
    "report": {"top_k": len(variant_plans)},
    "base_overrides": base_overrides,
    "candidates": [
        {"name": item["name"], "params": item["params"]}
        for item in variant_plans
    ],
}

generated_config_path.parent.mkdir(parents=True, exist_ok=True)
generated_config_path.write_text(
    yaml.safe_dump(tuning_config, sort_keys=False),
    encoding="utf-8",
)

runner_cmd = [
    PYTHON_BIN,
    "-u",
    "tuning/run_pdhg_exact_list.py",
    "--config",
    generated_config_path.as_posix(),
]
if int(MAX_CANDIDATES) > 0:
    runner_cmd.extend(["--max-candidates", str(int(MAX_CANDIDATES))])

ablation_context = {
    "study_tag": study_tag,
    "study_slug": study_slug,
    "generated_config_path": generated_config_path.as_posix(),
    "study_output_root": study_output_root.as_posix(),
    "launcher_log_path": launcher_log_path.as_posix(),
    "launcher_pid_path": launcher_pid_path.as_posix(),
    "runner_cmd": runner_cmd,
    "candidate_count": len(variant_plans),
}

print(json.dumps(ablation_context, indent=2))
print()
print(generated_config_path.read_text(encoding="utf-8"))

In [ ]:
#@title Validate every Hydra candidate without loading the model
import subprocess
from pathlib import Path

import hydra

config_dir = str((Path(REPO_DIR) / "configs").resolve())
config_name = Path(CONFIG_NAME).stem
validation_errors = []

with hydra.initialize_config_dir(version_base="1.3", config_dir=config_dir):
    for index, candidate in enumerate(tuning_config["candidates"]):
        overrides = list(tuning_config["base_overrides"])
        overrides.extend([
            f"{key}={str(value).lower() if isinstance(value, bool) else value}"
            for key, value in candidate["params"].items()
        ])
        try:
            hydra.compose(config_name=config_name, overrides=overrides)
        except Exception as exc:
            validation_errors.append((index, candidate["name"], str(exc)))

if validation_errors:
    for error in validation_errors:
        print(error)
    raise RuntimeError(f"{len(validation_errors)} Hydra candidates failed validation.")

dry_run_cmd = list(ablation_context["runner_cmd"]) + ["--dry-run"]
dry_run = subprocess.run(
    dry_run_cmd,
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(dry_run.stdout)
if dry_run.returncode != 0:
    raise RuntimeError(f"Exact-list runner dry-run failed with code {dry_run.returncode}.")
print(f"Validated {len(tuning_config['candidates'])} candidates.")

In [ ]:
#@title Launch the sequential ablation in the background
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
log_path = Path(ablation_context["launcher_log_path"])
pid_path = Path(ablation_context["launcher_pid_path"])
log_path.parent.mkdir(parents=True, exist_ok=True)

with log_path.open("w", encoding="utf-8") as log_handle:
    process = subprocess.Popen(
        ablation_context["runner_cmd"],
        cwd=REPO_DIR,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )

pid_path.write_text(str(process.pid), encoding="utf-8")
print(f"PID: {process.pid}")
print(f"Candidates: {ablation_context['candidate_count']}")
print(f"Launcher log: {log_path}")
print(f"Study root: {ablation_context['study_output_root']}")

In [ ]:
#@title Show ablation status
import json
import os
from pathlib import Path

pid_path = Path(ablation_context["launcher_pid_path"])
pid = int(pid_path.read_text(encoding="utf-8")) if pid_path.exists() else None
running = False
if pid is not None:
    try:
        os.kill(pid, 0)
        running = True
    except OSError:
        running = False
print({"pid": pid, "process_visible": running})

study_root = Path(ablation_context["study_output_root"]) / ablation_context["study_slug"]
study_dirs = sorted(
    [path for path in study_root.glob("*") if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
)
latest_study_dir = study_dirs[-1] if study_dirs else None
print(f"Latest study: {latest_study_dir}")

if latest_study_dir is not None:
    progress_path = latest_study_dir / "progress.json"
    current_path = latest_study_dir / "current_candidate.json"
    if progress_path.exists():
        print(json.dumps(json.loads(progress_path.read_text(encoding="utf-8")), indent=2))
    if current_path.exists():
        print()
        print("Current candidate:")
        print(json.dumps(json.loads(current_path.read_text(encoding="utf-8")), indent=2))

In [ ]:
#@title Show launcher and current-candidate logs
import json
from pathlib import Path

launcher_log = Path(ablation_context["launcher_log_path"])
if launcher_log.exists():
    lines = launcher_log.read_text(encoding="utf-8", errors="ignore").splitlines()
    print("=== Launcher log ===")
    print("\n".join(lines[-int(LOG_TAIL_LINES):]) if lines else "<empty>")
else:
    print(f"Launcher log not found: {launcher_log}")

if latest_study_dir is not None:
    current_path = latest_study_dir / "current_candidate.json"
    if current_path.exists():
        current = json.loads(current_path.read_text(encoding="utf-8"))
        current_log = Path(current["launcher_log"])
        if current_log.exists():
            lines = current_log.read_text(encoding="utf-8", errors="ignore").splitlines()
            print()
            print(f"=== Current candidate: {current.get('candidate_name')} ===")
            print("\n".join(lines[-int(LOG_TAIL_LINES):]) if lines else "<empty>")

In [ ]:
#@title Show the tail-length and tau0 leaderboard
import json
from pathlib import Path

if latest_study_dir is None:
    raise FileNotFoundError("No study directory exists yet.")

leaderboard_path = latest_study_dir / "leaderboard.json"
if not leaderboard_path.exists():
    raise FileNotFoundError(f"Leaderboard not found yet: {leaderboard_path}")

rows = json.loads(leaderboard_path.read_text(encoding="utf-8"))
reference_row = next((row for row in rows if row.get("candidate_name") == "reference"), None)
if reference_row is not None:
    for row in rows:
        for metric_key in ["psnr_max_mean", "ssim_max_mean", "lpips_min_mean"]:
            value = row.get(metric_key)
            reference_value = reference_row.get(metric_key)
            if isinstance(value, (int, float)) and isinstance(reference_value, (int, float)):
                row[f"delta_{metric_key}"] = value - reference_value

columns = [
    "candidate_index",
    "candidate_name",
    "status",
    "score",
    "psnr_max_mean",
    "ssim_max_mean",
    "lpips_min_mean",
    "delta_psnr_max_mean",
    "delta_ssim_max_mean",
    "delta_lpips_min_mean",
    "sampler.annealing_scheduler_config.theorem1_tail_steps",
    "inverse_task.admm_config.pdhg.tau",
    "sampler.annealing_scheduler_config.theorem1_tail_lambda",
    "++inverse_task.admm_config.early_stop",
]

try:
    import pandas as pd
    frame = pd.DataFrame(rows)
    display(frame[[column for column in columns if column in frame.columns]])
except ImportError:
    print(json.dumps(rows, indent=2))

print(f"Fixed c in every run: {REFERENCE_TAIL_C}")
print(f"Fixed raw rho power in every run: {REFERENCE_RHO_POWER}")
print(f"Leaderboard JSON: {leaderboard_path}")
print(f"Leaderboard CSV: {latest_study_dir / 'leaderboard.csv'}")

In [ ]:
#@title Copy the ablation study to Google Drive
import shutil
from pathlib import Path

if latest_study_dir is None:
    raise FileNotFoundError("No study directory exists yet.")

export_root = Path(DRIVE_EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)
destination = export_root / latest_study_dir.name
if destination.exists():
    shutil.rmtree(destination)
shutil.copytree(latest_study_dir, destination)

for source_text in [
    ablation_context["generated_config_path"],
    ablation_context["launcher_log_path"],
]:
    source = Path(source_text)
    if source.exists():
        shutil.copy2(source, export_root / source.name)

print(f"Copied study to: {destination}")